# No.5 圧縮センシング — Wavelet版

ランダムサンプリングされたMRI k空間データから、ウェーブレット収縮を用いて画像を再構成します。

**アルゴリズム:**
1. ウェーブレット変換
2. ハード閾値処理（スパース化）
3. 逆ウェーブレット変換
4. k空間データ整合性の強制
5. 閾値を段階的に下げながら繰り返す

In [ ]:
import numpy as np
import pywt
import scipy.io
import skimage.io
import skimage.metrics
import matplotlib.pyplot as plt
import japanize_matplotlib


In [ ]:
# パラメータ
J = 3
wavelet = 'db3'
T = 0.2
step = 40

# 画像とマスクの読み込み
img = skimage.io.imread('data/MRI05.pgm').astype(float) / 255.0
mask = skimage.io.imread('data/mask_1d_rand40.tiff').astype(bool)
n = img.shape[0]
if mask.shape != img.shape:
    mask = mask[:n, :n]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(img, cmap='gray')
axes[0].set_title('Original MRI')
axes[0].axis('off')
axes[1].imshow(mask, cmap='gray')
axes[1].set_title(f'Sampling mask ({mask.mean()*100:.0f}%)')
axes[1].axis('off')
plt.show()

In [ ]:
# k空間の部分サンプリング
Kfull = np.fft.fftshift(np.fft.fft2(np.fft.fftshift(img)))
Kobs = Kfull * mask

def apply_threshold(coeffs, threshold):
    new_coeffs = [coeffs[0]]
    for detail in coeffs[1:]:
        new_coeffs.append(tuple(c * (np.abs(c) >= threshold) for c in detail))
    return new_coeffs

# 圧縮センシング再構成ループ
sk = np.zeros_like(img)
psnr_history = []

for s in range(step):
    coeffs = pywt.wavedec2(sk, wavelet, level=J, mode='periodization')
    coeffs = apply_threshold(coeffs, T)
    sk = pywt.waverec2(coeffs, wavelet, mode='periodization')
    Krec = np.fft.fftshift(np.fft.fft2(np.fft.fftshift(sk)))
    Krec[mask] = Kobs[mask]
    sk = np.real(np.fft.fftshift(np.fft.ifft2(np.fft.fftshift(Krec))))
    T *= 0.9
    psnr_history.append(skimage.metrics.peak_signal_noise_ratio(img, sk, data_range=1.0))

plt.plot(psnr_history)
plt.xlabel('Iteration')
plt.ylabel('PSNR [dB]')
plt.title('PSNR vs Iteration')
plt.grid(True)
plt.show()

In [ ]:
psnr_final = skimage.metrics.peak_signal_noise_ratio(img, sk, data_range=1.0)
ssim_final = skimage.metrics.structural_similarity(img, sk, data_range=1.0)

sk_zf = np.real(np.fft.fftshift(np.fft.ifft2(np.fft.fftshift(Kobs))))
psnr_zf = skimage.metrics.peak_signal_noise_ratio(img, sk_zf, data_range=1.0)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Original')
axes[0].axis('off')
axes[1].imshow(np.clip(sk_zf, 0, 1), cmap='gray', vmin=0, vmax=1)
axes[1].set_title(f'Zero-fill (PSNR={psnr_zf:.1f}dB)')
axes[1].axis('off')
axes[2].imshow(np.clip(sk, 0, 1), cmap='gray', vmin=0, vmax=1)
axes[2].set_title(f'Wavelet CS (PSNR={psnr_final:.1f}dB, SSIM={ssim_final:.3f})')
axes[2].axis('off')
fig.tight_layout()
plt.show()